# 09 Rerun Knowledge-Enhanced BioBART With Optimized Graph



## 1. Imports and Configuration



In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

import evaluate
# Disable hf_transfer unless the package is explicitly installed.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
MODEL_NAME = "GanjinZero/biobart-v2-base"
GRAPH_VERSION = "balanced"  
MAX_TERMS = 3
RUN_TEST = True

GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "train_clean.csv"
VAL_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "val_clean.csv"
TEST_PATH = PROJECT_ROOT / "data" / "sentence_no_context" / "test_clean.csv"

MAPPING_PATHS = {
    "high_precision": PROJECT_ROOT / "data" / "knowledge_graph" / "mappings_high_precision.json",
    "balanced": PROJECT_ROOT / "data" / "knowledge_graph" / "mappings_balanced.json",
    "high_coverage": PROJECT_ROOT / "data" / "knowledge_graph" / "mappings_high_coverage.json",
}

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "models" / "biobart_sentence_no_context" / "best_model",
    PROJECT_ROOT / "models" / "biobart_sentence_no_context",
    PROJECT_ROOT / "models" / "biobart_knowledge_enhanced" / "best_model",
    PROJECT_ROOT / "models" / "biobart_knowledge_enhanced",
]

RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobart_optimized_kg_predictions.csv"
VAL_PREDICTION_PATH = RESULTS_DIR / "biobart_optimized_kg_validation_predictions.csv"
METRICS_PATH = RESULTS_DIR / "biobart_optimized_kg_metrics.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 180)

print(f"Project root: {PROJECT_ROOT}")
print(f"Graph version: {GRAPH_VERSION}")
print(f"Max mappings per prompt: {MAX_TERMS}")
print(f"Generation config: {GENERATION_CONFIG}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Load Data

All splits are loaded for coverage analysis. Validation is generated first; test generation is controlled by `RUN_TEST`.

In [ ]:
REQUIRED_COLUMNS = ["pair_id", "sent_id", "label", "complex", "simple"]

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")
    df = pd.read_csv(path)
    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"{split_name} is missing columns: {missing}")
    df = df[REQUIRED_COLUMNS].copy()
    for column in ["complex", "simple"]:
        df[column] = df[column].fillna("").astype(str).str.strip()
    df = df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)
    return df

train_df = load_split(TRAIN_PATH, "train")
val_df = load_split(VAL_PATH, "validation")
test_df = load_split(TEST_PATH, "test")

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")
display(train_df.head())

## 3. Load Optimized Graph Mappings

The mapping JSON contains automatically induced source-to-target term simplifications selected in notebook 08.

In [ ]:
if GRAPH_VERSION not in MAPPING_PATHS:
    raise ValueError(f"Unknown GRAPH_VERSION={GRAPH_VERSION}. Choose one of {list(MAPPING_PATHS)}")

MAPPING_PATH = MAPPING_PATHS[GRAPH_VERSION]
if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"Missing mapping file: {MAPPING_PATH}. Run notebook 08 first.")

with open(MAPPING_PATH, "r", encoding="utf-8") as file:
    term_to_simple: dict[str, str] = json.load(file)

print(f"Loaded mapping file: {MAPPING_PATH.relative_to(PROJECT_ROOT)}")
print(f"Mappings: {len(term_to_simple):,}")
display(pd.DataFrame([{"source": key, "target": value} for key, value in list(term_to_simple.items())[:50]]))

## 4. Build Knowledge-Enhanced Inputs

Term detection uses lowercase word-boundary matching, simple singular/plural variants, longer-term priority, and overlap avoidance. We insert at most three mappings to reduce prompt noise.

In [ ]:
def normalize_term(text: Any) -> str:
    text = str(text).lower().replace("_", " ").strip()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def singularize_token(token: str) -> str:
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token

def singularize_phrase(phrase: str) -> str:
    return " ".join(singularize_token(token) for token in normalize_term(phrase).split())

def plural_variants(term: str) -> set[str]:
    normalized = normalize_term(term)
    singular = singularize_phrase(normalized)
    variants = {normalized, singular}
    tokens = singular.split()
    if tokens:
        last = tokens[-1]
        plural_last = last[:-1] + "ies" if last.endswith("y") else last + "s"
        variants.add(" ".join(tokens[:-1] + [plural_last]))
    return {variant for variant in variants if variant}

def find_graph_terms(sentence: str, mapping: dict[str, str], max_terms: int = MAX_TERMS) -> list[tuple[str, str]]:
    sentence_norm = normalize_term(sentence)
    singular_sentence = " ".join(singularize_token(token) for token in sentence_norm.split())
    matches: list[tuple[str, str]] = []
    occupied_spans: list[tuple[int, int]] = []

    for source, target in sorted(mapping.items(), key=lambda item: (len(item[0].split()), len(item[0])), reverse=True):
        candidate_match = None
        for variant in sorted(plural_variants(source), key=len, reverse=True):
            pattern = rf"(?<![a-z0-9]){re.escape(variant)}(?![a-z0-9])"
            candidate_match = re.search(pattern, sentence_norm) or re.search(pattern, singular_sentence)
            if candidate_match:
                break
        if not candidate_match:
            continue
        span = candidate_match.span()
        if any(not (span[1] <= start or span[0] >= end) for start, end in occupied_spans):
            continue
        matches.append((candidate_match.group(0), target))
        occupied_spans.append(span)
        if len(matches) >= max_terms:
            break
    return matches

PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""


def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(complex_sentence).strip())


def build_knowledge_prompt(sentence: str, term_pairs: list[tuple[str, str]]) -> str:
    sentence = str(sentence).strip()
    if not term_pairs:
        return build_prompt(sentence)

    mapping_lines = "\n".join(f"- {source} = {target}" for source, target in term_pairs[:MAX_TERMS])
    return f"""You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Use these term simplifications if relevant:
{mapping_lines}

Sentence:
{sentence}

Simplified sentence:"""

def add_knowledge_inputs(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    detected = enriched["complex"].map(lambda sentence: find_graph_terms(sentence, term_to_simple, max_terms=MAX_TERMS))
    enriched["detected_mappings"] = detected.map(lambda pairs: "; ".join(f"{source}->{target}" for source, target in pairs))
    enriched["detected_mappings_count"] = detected.map(len)
    enriched["knowledge_input"] = [
        build_knowledge_prompt(sentence, pairs)
        for sentence, pairs in zip(enriched["complex"].tolist(), detected.tolist(), strict=False)
    ]
    return enriched

train_enriched_df = add_knowledge_inputs(train_df)
val_enriched_df = add_knowledge_inputs(val_df)
test_enriched_df = add_knowledge_inputs(test_df)

for split_name, df in [("train", train_enriched_df), ("validation", val_enriched_df), ("test", test_enriched_df)]:
    coverage = df["detected_mappings_count"].gt(0).mean()
    avg_mappings = df["detected_mappings_count"].mean()
    print(f"{split_name}: {coverage:.2%} coverage; average mappings/example = {avg_mappings:.3f}")

display(val_enriched_df[["complex", "simple", "detected_mappings", "knowledge_input"]].head())



## 5. Coverage and Top Detected Mappings

Coverage is reported for all splits, but graph selection should be based on validation behavior, not test tuning.

In [ ]:
def mapping_counter(df: pd.DataFrame) -> Counter[str]:
    counter: Counter[str] = Counter()
    for value in df["detected_mappings"].fillna(""):
        if not value:
            continue
        counter.update(value.split("; "))
    return counter

coverage_rows = []
for split_name, df in [("train", train_enriched_df), ("validation", val_enriched_df), ("test", test_enriched_df)]:
    with_terms = int(df["detected_mappings_count"].gt(0).sum())
    coverage_rows.append({
        "split": split_name,
        "rows": len(df),
        "examples_with_kg_terms": with_terms,
        "coverage_pct": round(100 * with_terms / len(df), 2),
        "avg_mappings_per_example": round(float(df["detected_mappings_count"].mean()), 3),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

print("Top validation detected mappings")
display(pd.DataFrame(mapping_counter(val_enriched_df).most_common(30), columns=["mapping", "count"]))

print("Top test detected mappings")
display(pd.DataFrame(mapping_counter(test_enriched_df).most_common(30), columns=["mapping", "count"]))

## 6. Load Fine-Tuned BioBART

The notebook prefers a local fine-tuned checkpoint. If no checkpoint is found, it falls back to `GanjinZero/biobart-v2-base`, but that is not the desired comparison for the final experiment.

In [ ]:
def has_model_files(path: Path) -> bool:
    return path.exists() and (path / "config.json").exists()

local_checkpoint = next((path for path in CHECKPOINT_CANDIDATES if has_model_files(path)), None)
model_source = str(local_checkpoint) if local_checkpoint is not None else MODEL_NAME
tokenizer_source = model_source if local_checkpoint is not None and (Path(model_source) / "tokenizer_config.json").exists() else MODEL_NAME

if local_checkpoint is None:
    print("WARNING: No local fine-tuned BioBART checkpoint found. Falling back to the pretrained base model.")
else:
    print(f"Loading fine-tuned checkpoint: {Path(model_source).relative_to(PROJECT_ROOT)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
model = AutoModelForSeq2SeqLM.from_pretrained(model_source)
model.to(device)
model.eval()

print(f"Tokenizer source: {tokenizer_source}")
print(f"Model source: {model_source}")
print(f"Device: {device}")

## 7. Generate Validation Predictions First

Validation generation checks that the optimized prompt and model checkpoint work before running the test set.

In [ ]:
def clean_prediction(text: str) -> str:
    """Remove prompt echoes and generation boilerplate from decoded text."""
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return ""

    # Decoder-only or poorly adapted seq2seq checkpoints can echo the input prompt.
    prompt_markers = [
        "Simplified sentence:",
        "Rewrite the biomedical sentence for a general audience.",
        "Rewrite this biomedical sentence in simpler language:",
        "Simplify the biomedical sentence.",
        "Sentence:",
    ]
    for marker in prompt_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    prefixes = ["Simplified:", "Answer:", "Prediction:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text).strip()

def generate_predictions(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    output = df.copy()
    predictions: list[str] = []
    inputs = output["knowledge_input"].tolist()

    for start in tqdm(range(0, len(inputs), batch_size), desc="Generating"):
        batch_inputs = inputs[start:start + batch_size]
        try:
            encoded = tokenizer(
                batch_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)
            with torch.no_grad():
                generated = model.generate(**encoded, **GENERATION_CONFIG)
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
            predictions.extend([clean_prediction(text) or "" for text in decoded])
        except Exception as exc:
            print(f"Generation failed for batch starting at {start}: {exc}")
            predictions.extend([""] * len(batch_inputs))
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    output["prediction"] = predictions
    return output

val_prediction_df = generate_predictions(val_enriched_df, batch_size=8)
val_prediction_df.to_csv(VAL_PREDICTION_PATH, index=False)
print(f"Saved validation predictions to: {VAL_PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(val_prediction_df[["complex", "detected_mappings", "simple", "prediction"]].head())



## 8. Metric Functions

BLEU is reported on the SacreBLEU 0-100 scale, matching the later BioBART notebooks.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

val_metrics_df = compute_metrics(val_prediction_df)
display(val_metrics_df)


## 9. Generate Test Predictions

Run this only after validation generation works. Set `RUN_TEST = False` in the configuration cell to skip test inference.

In [ ]:
if RUN_TEST:
    test_prediction_df = generate_predictions(test_enriched_df, batch_size=8)
    test_prediction_df.to_csv(PREDICTION_PATH, index=False)
    print(f"Saved test predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
else:
    test_prediction_df = pd.DataFrame()
    print("RUN_TEST=False, skipping test generation.")

if len(test_prediction_df):
    display(test_prediction_df[["complex", "detected_mappings", "simple", "prediction"]].head())

## 10. Test and Subset Evaluation

Metrics are computed for all test examples, examples with detected KG terms, and examples without detected KG terms.

In [ ]:
def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")

if len(test_prediction_df):
    subsets = {
        "all_test": test_prediction_df,
        "with_kg_terms": test_prediction_df[test_prediction_df["detected_mappings_count"].gt(0)].copy(),
        "without_kg_terms": test_prediction_df[test_prediction_df["detected_mappings_count"].eq(0)].copy(),
    }
    metric_rows = []
    for subset_name, subset_df in subsets.items():
        subset_metrics = compute_metrics(subset_df) if len(subset_df) else pd.DataFrame()
        metric_rows.append({
            "subset": subset_name,
            "rows": len(subset_df),
            "coverage_pct": round(100 * len(subset_df) / len(test_prediction_df), 2),
            "avg_mappings_per_example": round(float(subset_df["detected_mappings_count"].mean()), 3) if len(subset_df) else 0.0,
            "SARI": metric_value(subset_metrics, "SARI") if len(subset_df) else np.nan,
            "BLEU": metric_value(subset_metrics, "BLEU") if len(subset_df) else np.nan,
            "BERTScore Precision": metric_value(subset_metrics, "BERTScore Precision") if len(subset_df) else np.nan,
            "BERTScore Recall": metric_value(subset_metrics, "BERTScore Recall") if len(subset_df) else np.nan,
            "BERTScore F1": metric_value(subset_metrics, "BERTScore F1") if len(subset_df) else np.nan,
        })
    test_metrics_df = pd.DataFrame(metric_rows)
    test_metrics_df.to_csv(METRICS_PATH, index=False)
    print(f"Saved metrics to: {METRICS_PATH.relative_to(PROJECT_ROOT)}")
    display(test_metrics_df)
else:
    test_metrics_df = pd.DataFrame()
    print("No test predictions available for subset evaluation.")

## 11. Compare Against Baselines



In [ ]:
if len(test_metrics_df):
    all_row = test_metrics_df[test_metrics_df["subset"].eq("all_test")].iloc[0]
    comparison_df = pd.DataFrame([
        {"Experiment": "E2", "Model": "BioBART direct", "SARI": 31.86, "BLEU": 30.91, "BERTScore F1": 0.932},
        {"Experiment": "E6", "Model": "KG-BioBART original", "SARI": 31.63, "BLEU": 32.22, "BERTScore F1": 0.928},
        {
            "Experiment": "E7",
            "Model": f"KG-BioBART optimized graph ({GRAPH_VERSION})",
            "SARI": all_row["SARI"],
            "BLEU": all_row["BLEU"],
            "BERTScore F1": all_row["BERTScore F1"],
        },
    ])
    display(comparison_df)
else:
    comparison_df = pd.DataFrame()
    print("Run test generation to build the comparison table.")

## 12. Qualitative Analysis



In [ ]:
if len(test_prediction_df):
    kg_examples = test_prediction_df[test_prediction_df["detected_mappings_count"].gt(0)].sample(
        n=min(30, int(test_prediction_df["detected_mappings_count"].gt(0).sum())),
        random_state=SEED,
    )
    display(kg_examples[["complex", "detected_mappings", "simple", "prediction"]])
else:
    print("No test predictions available.")

In [ ]:
DIRECT_PREDICTION_CANDIDATES = [
    RESULTS_DIR / "biobart_sentence_no_context_predictions.csv",
    RESULTS_DIR / "biobart_predictions.csv",
    RESULTS_DIR / "biobart_test_predictions.csv",
]

direct_prediction_path = next((path for path in DIRECT_PREDICTION_CANDIDATES if path.exists()), None)
if len(test_prediction_df) and direct_prediction_path is not None:
    direct_df = pd.read_csv(direct_prediction_path)
    prediction_col = "prediction" if "prediction" in direct_df.columns else None
    if prediction_col is None:
        print(f"Found {direct_prediction_path.name}, but no prediction column is available.")
    else:
        key_columns = [column for column in ["pair_id", "sent_id"] if column in direct_df.columns and column in test_prediction_df.columns]
        if key_columns:
            merged = test_prediction_df.merge(
                direct_df[key_columns + [prediction_col]].rename(columns={prediction_col: "direct_prediction"}),
                on=key_columns,
                how="left",
            )
        else:
            merged = test_prediction_df.copy()
            merged["direct_prediction"] = direct_df[prediction_col].reindex(range(len(merged))).tolist()
        changed = merged[
            merged["direct_prediction"].fillna("").astype(str).str.strip().ne(merged["prediction"].fillna("").astype(str).str.strip())
            & merged["detected_mappings_count"].gt(0)
        ].copy()
        display(changed[["complex", "detected_mappings", "simple", "direct_prediction", "prediction"]].head(20))
elif direct_prediction_path is None:
    print("No direct BioBART prediction file found, so help/hurt comparison is skipped.")
else:
    print("No optimized KG predictions available for direct comparison.")

## 13. Audit: Subset Metrics and KG Effect

This section investigates counterintuitive subset results. It does not rerun BioBART generation. It reloads saved predictions, verifies subset membership, recomputes metrics independently, checks whether KG prompts were actually used, and compares with direct BioBART predictions when available.

In [ ]:
AUDIT_PREDICTION_PATH = PREDICTION_PATH
if not AUDIT_PREDICTION_PATH.exists():
    raise FileNotFoundError(f"Missing optimized KG predictions: {AUDIT_PREDICTION_PATH}. Run test generation first or copy the CSV into results/.")

audit_df = pd.read_csv(AUDIT_PREDICTION_PATH)
for column in ["complex", "simple", "prediction", "detected_mappings"]:
    if column in audit_df.columns:
        audit_df[column] = audit_df[column].fillna("").astype(str)
if "detected_mappings_count" not in audit_df.columns:
    audit_df["detected_mappings_count"] = audit_df["detected_mappings"].map(lambda value: 0 if not str(value).strip() else len(str(value).split("; ")))

print(f"Loaded audit predictions: {AUDIT_PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(audit_df):,}")
print(f"Columns: {list(audit_df.columns)}")
display(audit_df.head())

In [ ]:
expected_test = test_enriched_df.copy()
identity_columns = [column for column in ["pair_id", "sent_id", "complex", "simple"] if column in audit_df.columns and column in expected_test.columns]

identity_report = {
    "prediction_rows": len(audit_df),
    "current_test_rows": len(expected_test),
    "same_row_count": len(audit_df) == len(expected_test),
}
for column in identity_columns:
    identity_report[f"same_{column}"] = bool(audit_df[column].astype(str).reset_index(drop=True).eq(expected_test[column].astype(str).reset_index(drop=True)).all())

display(pd.DataFrame([identity_report]))

subset_report = pd.DataFrame([
    {
        "subset": "all_test",
        "rows": len(audit_df),
        "empty_predictions": int(audit_df["prediction"].str.strip().eq("").sum()),
        "examples_with_kg_terms": int(audit_df["detected_mappings_count"].gt(0).sum()),
        "coverage_pct": round(100 * audit_df["detected_mappings_count"].gt(0).mean(), 2),
        "avg_mappings_per_example": round(float(audit_df["detected_mappings_count"].mean()), 3),
    },
    {
        "subset": "with_kg_terms",
        "rows": int(audit_df["detected_mappings_count"].gt(0).sum()),
        "empty_predictions": int(audit_df.loc[audit_df["detected_mappings_count"].gt(0), "prediction"].str.strip().eq("").sum()),
        "examples_with_kg_terms": int(audit_df["detected_mappings_count"].gt(0).sum()),
        "coverage_pct": round(100 * audit_df["detected_mappings_count"].gt(0).mean(), 2),
        "avg_mappings_per_example": round(float(audit_df.loc[audit_df["detected_mappings_count"].gt(0), "detected_mappings_count"].mean()), 3),
    },
    {
        "subset": "without_kg_terms",
        "rows": int(audit_df["detected_mappings_count"].eq(0).sum()),
        "empty_predictions": int(audit_df.loc[audit_df["detected_mappings_count"].eq(0), "prediction"].str.strip().eq("").sum()),
        "examples_with_kg_terms": 0,
        "coverage_pct": round(100 * audit_df["detected_mappings_count"].eq(0).mean(), 2),
        "avg_mappings_per_example": 0.0,
    },
])
display(subset_report)

print("Top detected mappings in audited predictions")
display(pd.DataFrame(mapping_counter(audit_df).most_common(30), columns=["mapping", "count"]))

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

def compute_metrics_with_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Shared metrics plus row counts for audit tables."""
    metrics = compute_metrics(df).copy()
    metric_df = df[["complex", "simple", "prediction"]].copy()
    for column in ["complex", "simple", "prediction"]:
        metric_df[column] = metric_df[column].fillna("").astype(str)
    bertscore_examples = int(metric_df[metric_df["prediction"].str.strip().ne("")].shape[0])
    count_rows = pd.DataFrame([
        {"metric": "sari_bleu_examples", "score": len(metric_df)},
        {"metric": "bertscore_examples", "score": bertscore_examples},
    ])
    return pd.concat([metrics, count_rows], ignore_index=True)


audit_subsets = {
    "all_test": audit_df,
    "with_kg_terms": audit_df[audit_df["detected_mappings_count"].gt(0)].copy(),
    "without_kg_terms": audit_df[audit_df["detected_mappings_count"].eq(0)].copy(),
}


audit_metric_rows = []
for subset_name, subset_df in audit_subsets.items():
    shared_metrics = compute_metrics_with_counts(subset_df)
    audit_metric_rows.append({
        "subset": subset_name,
        "rows": len(subset_df),
        "SARI": metric_value(shared_metrics, "SARI"),
        "BLEU": metric_value(shared_metrics, "BLEU"),
        "BERTScore F1": metric_value(shared_metrics, "BERTScore F1"),
        "sari_bleu_examples": metric_value(shared_metrics, "sari_bleu_examples"),
        "bertscore_examples": metric_value(shared_metrics, "bertscore_examples"),
    })


audit_metrics_df = pd.DataFrame(audit_metric_rows)
display(audit_metrics_df)


In [ ]:
# Check whether rows marked with KG terms really contain mapping blocks in the prompt, when knowledge_input was saved.
if "knowledge_input" in audit_df.columns:
    prompt_check_df = audit_df.copy()
    prompt_check_df["has_mapping_block"] = prompt_check_df["knowledge_input"].fillna("").astype(str).str.contains("Use these term simplifications", regex=False)
    display(pd.crosstab(prompt_check_df["detected_mappings_count"].gt(0), prompt_check_df["has_mapping_block"], rownames=["detected_mappings_count > 0"], colnames=["prompt_has_mapping_block"]))
else:
    print("knowledge_input was not saved in the prediction CSV, so prompt block presence cannot be checked from disk.")

with_kg_audit_examples = audit_df[audit_df["detected_mappings_count"].gt(0)].sample(
    n=min(50, int(audit_df["detected_mappings_count"].gt(0).sum())),
    random_state=SEED,
)
display(with_kg_audit_examples[["complex", "detected_mappings", "simple", "prediction"]])

In [ ]:
direct_prediction_path = next((path for path in DIRECT_PREDICTION_CANDIDATES if path.exists()), None)
if direct_prediction_path is None:
    print("No direct BioBART prediction file found. Expected one of:")
    for path in DIRECT_PREDICTION_CANDIDATES:
        print("-", path.relative_to(PROJECT_ROOT))
else:
    direct_df = pd.read_csv(direct_prediction_path)
    if "prediction" not in direct_df.columns:
        raise ValueError(f"Direct prediction file has no prediction column: {direct_prediction_path}")
    direct_df["prediction"] = direct_df["prediction"].fillna("").astype(str)
    key_columns = [column for column in ["pair_id", "sent_id"] if column in direct_df.columns and column in with_kg_audit_examples.columns]
    if key_columns:
        compare_examples_df = with_kg_audit_examples.merge(
            direct_df[key_columns + ["prediction"]].rename(columns={"prediction": "direct_biobart_prediction"}),
            on=key_columns,
            how="left",
        )
    else:
        compare_examples_df = with_kg_audit_examples.copy()
        compare_examples_df["direct_biobart_prediction"] = direct_df["prediction"].reindex(compare_examples_df.index).tolist()
    display(compare_examples_df[["complex", "detected_mappings", "simple", "direct_biobart_prediction", "prediction"]])